In [6]:
# Script to update the existing Webmap with Map image data
# Searches the AGOL for webmap and Map image 
from arcgis.gis import GIS
from arcgis.map import Map
from arcgis.features import FeatureLayer

In [7]:
# Update the Utility and LAC areas
UTILITY = "ALECTRA"
LAC_AREAS_ALL = [
   "HN01"
]

In [8]:
def search_AGOL(query, feature_type):
    try:
        search_text = f" title:'{query}' AND type:'{feature_type}'"
        print(f"Searching for:{search_text}\n")
        search_results = gis.content.search(query=search_text, max_items=1)
        print(search_results)
        
        if not search_results:
            print(f"{query} - Layers Not Found in AGOL.")
            return None

        search_result_item = search_results[0]
        return search_result_item
    
    except Exception as e:
        print(f"Error Searching Layers: {e}")
        return None        

In [9]:
# Function to format the title text   
def Format_Title(title_text):
    text_part = title_text.split('_')
    formatted = []

    for i, part in enumerate(text_part):
        # If it's the last part and looks like a code (letters + numbers)
        if i == len(text_part) - 1:
            formatted.append(part.upper())
        else:
            formatted.append(part.capitalize())
            
    return " ".join(formatted)

In [12]:
def Update_Webmap(gis_conn, current_webmap, landbase_map):
    
    webmap_item = gis.content.get(current_webmap.id )
    layer_item = gis.content.get(landbase_map.id)
    
    # Get the internal definition (the JSON)
    webmap_definition = webmap_item.get_data()

    # Construct the layer object
    general_landbase_layer = {
        "id": f"General Landbase {layer_item.id}",
        "layerType": "ArcGISMapServiceLayer",
        "url": layer_item.url, # Path to the specific sublayer
        "visibility": True,
        "title": Format_Title(layer_item.title),
        "itemId": layer_item.id,
        "opacity": 1,
        "maxScale": 50000
    }

    # INSERT at index 0 (The bottom of the list)
    if 'operationalLayers' not in webmap_definition:
        webmap_definition['operationalLayers'] = []
    
    webmap_definition['operationalLayers'].insert(0, general_landbase_layer)
    
    # Update the Web Map item
    webmap_item.update(data=json.dumps(webmap_definition))
    print(f"Map '{new_layer_item.title}' has been updated to the {webmap_item.title}.")

In [13]:
try:
    gis = GIS("home")
    print(f"Successfully connected to {gis.properties.portalName} as {gis.properties.user.username}")
    # Search for Feature Layers (Hosted Feature Services)
    print("\n--- Searching for Layers ---")

    for lac in LAC_AREAS_ALL:
        # print(f"{UTILITY} {lac} WebMap")
        # webmap_update = search_AGOL(f"{UTILITY} {lac} WebMap", "Web Map")
        webmap_update = search_AGOL(f"TEST Data Webmap - HN01", "Web Map")            
        general_landbase = search_AGOL(f"GENLANDBASEON_LANDBASE__{lac}", "Map Service")

        # Function to update the webmap
        Update_Webmap(gis, webmap_update, general_landbase)          
        
except Exception as e:
    print(f"An error occurred: {e}")

Successfully connected to ArcGIS Enterprise as sameer.bajracharya@planview.ca

--- Searching for Layers ---

Searching for: title:'TEST Data Webmap - HN01' AND type:'Web Map'
[<Item title:"TEST Data Webmap - HN01" type:Web Map owner:sameer.bajracharya@planview.ca>]

Searching for: title:'GENLANDBASEON_LANDBASE__HN01' AND type:'Map Service'
[<Item title:"GENLANDBASEON_LANDBASE__HN01" type:Map Image Layer owner:miguel.vargas@planview.ca>]
Layer 'GENLANDBASEON_LANDBASE__HN01' has been updated to the bottom.


In [3]:
#  For testing
from arcgis.gis import GIS
import json

gis = GIS("home")

# 1. Get the Web Map and the New Layer
wm_item = gis.content.get("cfbf86de48e2495181fd92ce4ad64766")
new_layer_item = gis.content.get("71889ae6bbae445f8b4056d0528ef25d")

# 2. Get the current Map Data (JSON)
wm_data = wm_item.get_data()

# 3. Construct the layer object
general_landbase_layer = {
    "id": "AddedLayer_Bottom",
    "layerType": "ArcGISMapServiceLayer",
    "url": new_layer_item.url, # Path to the specific sublayer
    "visibility": True,
    "title": new_layer_item.title,
    "itemId": new_layer_item.id,
    "opacity": 1,
    "maxScale": 50000
}

# 4. INSERT at index 0 (The bottom of the list)
if 'operationalLayers' not in wm_data:
    wm_data['operationalLayers'] = []

# wm_data['operationalLayers'].insert(0, general_landbase_layer)

# 5. Update the Web Map item
# wm_item.update(data=json.dumps(wm_data))

# print(f"Layer '{new_layer_item.title}' has been moved to the bottom of the draw order.")

Layer 'GENLANDBASEON_LANDBASE__HN01' has been moved to the bottom of the draw order.
